# Test du wrapper MME_Global et méthodes générales


In [1]:
import sys
import os

project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

In [2]:
%load_ext autoreload
%autoreload 2
import os
import json
import pickle as pkl
from gbmhackathon.utils import global_wrapper

Load la config et effectue quelques modifications (j'arrive pas à effectuer les modifs dans sagemaker c'est relou)

In [3]:
path_config="../gbmhackathon/utils/config.json"
with open(path_config, 'r') as f:
    config= json.load(f)

print(config.keys())

output_dir = "/home/sagemaker-user/results"
os.makedirs(output_dir, exist_ok=True)
config['global_settings']["output_dir"]="/home/sagemaker-user/results"
config['global_settings']["device"]='cpu'
config["MME_Model"]["training"]["epochs"]=10
config["MME_Model"]["architecture"]["MME"]["wes_cfg"]["net_config"]["layers"]= [1790, 550, 64]
del config["MME_Model"]["modalities_data"]["modalities"]["spatial"] ## Marche pas à cause de timothée
#del config["MME_Model"]["modalities_data"]["modalities"]["wes"] ## Marche pas à cause de timothée
config["gbm_head"]["head_cfg"]["net_config"]["layers"]=[6,1024,512,5]
print(config)
config["Experiments"]["Multiple_run_experiment"]["params"]["n_run"]=5

dict_keys(['global_settings', 'MME_Model', 'gbm_head', 'Global_Architecture', 'Experiments'])
{'global_settings': {'device': 'cpu', 'output_dir': '/home/sagemaker-user/results'}, 'MME_Model': {'modalities_data': {'modalities': {'hne': 'embeddings_HnE_OptimusH0.pkl', 'clinical': '2025-03-30_14-23_clinical_emb_V1.pkl', 'wes': '2025-04-05_13-40_wes_emb_V1.pkl'}, 'pkl_storage_folder': 'embedding_V1', 'missing_mods': 's3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/missing_mod_per_samples.pkl'}, 'training': {'batch_size': 16, 'epochs': 10, 'InfoNCE_Loss': {'temperature': 0.05, 'similarity': 'nt-xent', 'alpha': 0.1, 'bound': -10, 'beta': 0.2, 'nce_eps': 1e-08, 'reg_eps': 1e-08, 'use_all_positives': False}, 'Optimizer': {'lr': 0.001}}, 'architecture': {'MME': {'hne_cfg': {'net_type': 'mlp', 'net_config': {'layers': [1536, 512, 64], 'dropout': 0.65, 'act_fn': 'torch.nn.ReLU', 'norm_layer': 'torch.nn.LayerNorm'}}, 'spatial_cfg': {'net_type': 'attention', 'net_config': 

On instancie la classe, avec la config 


In [ ]:
wahou=global_wrapper.MME_Global(config)


In [ ]:
wahou.dataset.__len__()


In [ ]:
print(wahou.mme)

pour l'entraîner : 

In [ ]:
wahou.fit_mme()

Pour sauvegarder : attention, tout est sauvegardé dans le dossier spécifié en config (voir ligne : config['global_settings']["output_dir"]="/home/sagemaker-user/results"  au dessu)
Cela va sauvegarder une instance de la classe MME_Global, et son modèle dans un fichier séparé
--> De la sorte, on peut tout récupérer et savoir ce qui a été fait 

In [4]:
wahou.save()


NameError: name 'wahou' is not defined

In [ ]:
print(wahou.mme)

si on veut récupérer l'ensemble à postériorie : 

In [ ]:
with open("path/to/MME_global.pkl", "rb") as f:
    old_MME = pkl.load(f)
print("model : ", old_MME.mme)

In [ ]:
old_MME.reload_model()
print("now, model : ",old_MME.mme)



Essais sur wrapper GBM Net

In [4]:
youhou=global_wrapper.ModularModel(config)

['hne', 'clinical', 'wes']
hne_cfg
['hne', 'clinical', 'wes']
spatial_cfg
['hne', 'clinical', 'wes']
wes_cfg
['hne', 'clinical', 'wes']
clinical_cfg
['hne', 'clinical', 'wes']
bulk_cfg
['hne', 'clinical', 'wes']
sc_cfg


In [5]:
youhou.fit()

start phase 1 ...
Using device : cpu
device MME : cpu
hne torch.Size([16, 1536])
clinical torch.Size([16, 12])
wes torch.Size([16, 1790])
{'net_type': 'mlp', 'net_config': {'layers': [1536, 512, 64], 'dropout': 0.65, 'act_fn': <class 'torch.nn.modules.activation.ReLU'>, 'norm_layer': <class 'torch.nn.modules.normalization.LayerNorm'>}, 'device': 'cpu'}
Using device: cpu
No potential residual connections found
Using device: cpu
No potential residual connections found
Using device: cpu
No potential residual connections found


/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:25: ParameterNotFoundWarning: Parameter beta not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate', 'smoothing_func'].
  warnings.warn(
/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:25: ParameterNotFoundWarning: Parameter nce_eps not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate', 'smoothing_func'].
  warnings.warn(
/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:25: ParameterNotFoundWarning: Parameter reg_eps not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate', 'smoothing_func'].
  warnings.warn(


EPOCH 0 LOSS: 5.6151
EPOCH 1 LOSS: 6.5992
EPOCH 2 LOSS: 6.3437
EPOCH 3 LOSS: 6.8388
EPOCH 4 LOSS: 5.4972
EPOCH 5 LOSS: 6.7514
EPOCH 6 LOSS: 6.8119
EPOCH 7 LOSS: 6.1369
EPOCH 8 LOSS: 5.5161
EPOCH 9 LOSS: 6.5459
start phase 2 ...
cas 2 : device reconnu : cpu 
Using device: cpu
By keeping 35.00% of dropout augmented samples we went from:
54 dropout samples (32.14% dropout in dataset) -- to --> 21 dropout samples (15.56% dropout in dataset)
predictive loader (X_dict) device : cpu
predictive loader (batch_targets) device : cpu
Using device(prediction head) : cpu
No potential residual connections found


/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:25: ParameterNotFoundWarning: Parameter training not found in init arguments: ['self', 'head_cfg', 'mme', 'load_mme', 'mme_path', 'mme_cfg', 'freeze_mme'].
  warnings.warn(


epoch 0 - loss : 3.211851119995117
epoch 1 - loss : 2.1074059009552
epoch 2 - loss : 1.3009982109069824
epoch 3 - loss : 2.6585886478424072
epoch 4 - loss : 3.911377429962158
epoch 5 - loss : 0.9163137674331665
epoch 6 - loss : 2.1412858963012695
epoch 7 - loss : 2.2782158851623535
epoch 8 - loss : 1.2439631223678589
epoch 9 - loss : 1.034872055053711
epoch 10 - loss : 1.2393693923950195
epoch 11 - loss : 0.898966908454895
epoch 12 - loss : 2.1563732624053955
epoch 13 - loss : 1.5508955717086792
epoch 14 - loss : 0.7888434529304504
epoch 15 - loss : 1.2911899089813232
epoch 16 - loss : 2.0688071250915527
epoch 17 - loss : 0.6432842016220093
epoch 18 - loss : 0.8759818077087402
epoch 19 - loss : 1.099037528038025
epoch 20 - loss : 1.1164782047271729
epoch 21 - loss : 0.878771185874939
epoch 22 - loss : 1.4055078029632568
epoch 23 - loss : 1.671489953994751
epoch 24 - loss : 1.2449095249176025
epoch 25 - loss : 1.1278741359710693
epoch 26 - loss : 2.171715259552002
epoch 27 - loss : 3.41

In [ ]:
youhou.GbmNet